In [22]:
import openai 
import instructor 
from qdrant_client import QdrantClient
from pydantic import BaseModel, Field

In [23]:
def preprocess_review(review):
    return {
        'product_id': review['product_id'], 
        'main_category': review['main_category'],
        'average_rating': review['average_rating'],
        'rating_number': review['rating_number'],
        'features': review['features'],
        'description': review['description'],
        'price': review['price'],
        'images': review['images'],
        'videos': review['videos'],
        'store': review['store'],
        'categories': review['categories'],
        'details': review['details'],
        'parent_asin': review['parent_asin'],
    }

In [24]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Retrieve API keys from environment variables
openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GEMINI_API_KEY')
qdrant_url = os.getenv('QDRANT_URL')
qdrant_api_key = os.getenv('QDRANT_API_KEY')
langsmith_api_key = os.getenv('LANGSMITH_API_KEY')
if qdrant_url and "qdrant:6333" in qdrant_url:
    # Docker service host is not resolvable from a local notebook kernel
    qdrant_url = qdrant_url.replace("qdrant:6333", "localhost:6333")

# Verify keys are loaded
print(f"OpenAI API Key present: {bool(openai_api_key)}")
print(f"Google API Key present: {bool(google_api_key)}")
print(f"Qdrant URL present: {bool(qdrant_url)}")
print(f"Qdrant API Key present: {bool(qdrant_api_key)}")
print(f"Langsmith API Key present: {bool(langsmith_api_key)}")

OpenAI API Key present: True
Google API Key present: False
Qdrant URL present: True
Qdrant API Key present: False
Langsmith API Key present: True


In [25]:
client = instructor.from_openai(openai.OpenAI())

In [26]:
class RagGenerationResponse(BaseModel):
    answer: str = Field(description="The answer to the question")
    reasoning: str = Field(description="The reasoning behind the answer")

In [27]:
qdrant_client = QdrantClient(
    url="http://localhost:6333",
)

In [28]:
def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )
    return response.data[0].embedding


def retrieve_data(query, qdrant_client=qdrant_client, k=5):
    query_embedding = get_embedding(query)
    results = qdrant_client.query_points(
        collection_name="Amazon_Electronics_Data_Collection",
        query=query_embedding,
        limit=k,
        with_payload=True,
    )

    print("results.points: ", results.points)

    retrieved_context_ids = []
    retrieve_context = []
    similarity_scores = []
    retrieved_context_rating_numbers = []
    retrieve_context_main_categories = []
    retrieve_context_prices = []
    retrieve_context_images = []
    retrieve_context_videos = []
    retrieve_context_stores = []
    retrieve_context_categories = []
    retrieve_context_details = []
    retrieve_context_descriptions = []
    retrieve_context_features = []

    for result in results.points:
        payload = result.payload or {}
        retrieved_context_ids.append(result.id)
        similarity_scores.append(result.score)
        retrieve_context.append(payload.get('text', ''))
        retrieved_context_rating_numbers.append(payload.get('rating_number', None))
        retrieve_context_main_categories.append(payload.get('main_category', None))
        retrieve_context_prices.append(payload.get('price', None))
        retrieve_context_images.append(payload.get('images', None))
        retrieve_context_videos.append(payload.get('videos', None))
        retrieve_context_stores.append(payload.get('store', None))
        retrieve_context_categories.append(payload.get('categories', None))
        retrieve_context_details.append(payload.get('details', None))
        retrieve_context_descriptions.append(payload.get('description', None))
        retrieve_context_features.append(payload.get('features', None))

    return {
        'retrieved_context_ids': retrieved_context_ids,
        'retrieve_context': retrieve_context,
        'similarity_scores': similarity_scores,
        'retrieved_context_rating_numbers': retrieved_context_rating_numbers,
        'retrieve_context_main_categories': retrieve_context_main_categories,
        'retrieve_context_prices': retrieve_context_prices,
        'retrieve_context_images': retrieve_context_images,
        'retrieve_context_videos': retrieve_context_videos,
        'retrieve_context_stores': retrieve_context_stores,
        'retrieve_context_categories': retrieve_context_categories,
        'retrieve_context_details': retrieve_context_details,
        'retrieve_context_descriptions': retrieve_context_descriptions,
        'retrieve_context_features': retrieve_context_features,
    }

def process_context(context):
    formatted_context = []
    count = len(context.get('retrieved_context_ids', []))

    for index in range(count):
        point_id = context['retrieved_context_ids'][index]
        text = context['retrieve_context'][index] if index < len(context.get('retrieve_context', [])) else ''
        score = context['similarity_scores'][index] if index < len(context.get('similarity_scores', [])) else None
        rating_number = context['retrieved_context_rating_numbers'][index] if index < len(context.get('retrieved_context_rating_numbers', [])) else None
        main_category = context['retrieve_context_main_categories'][index] if index < len(context.get('retrieve_context_main_categories', [])) else ''
        price = context['retrieve_context_prices'][index] if index < len(context.get('retrieve_context_prices', [])) else None
        images = context['retrieve_context_images'][index] if index < len(context.get('retrieve_context_images', [])) else []
        videos = context['retrieve_context_videos'][index] if index < len(context.get('retrieve_context_videos', [])) else []
        store = context['retrieve_context_stores'][index] if index < len(context.get('retrieve_context_stores', [])) else ''
        categories = context['retrieve_context_categories'][index] if index < len(context.get('retrieve_context_categories', [])) else []
        details = context['retrieve_context_details'][index] if index < len(context.get('retrieve_context_details', [])) else ''
        description = context['retrieve_context_descriptions'][index] if index < len(context.get('retrieve_context_descriptions', [])) else ''
        features = context['retrieve_context_features'][index] if index < len(context.get('retrieve_context_features', [])) else []

        formatted_context.append(
            f"ID: {point_id}\n"
            f"Score: {score}\n"
            f"Rating Number: {rating_number}\n"
            f"Main Category: {main_category}\n"
            f"Price: {price}\n"
            f"Store: {store}\n"
            f"Categories: {categories}\n"
            f"Details: {details}\n"
            f"Description: {description}\n"
            f"Features: {features}\n"
            f"Images: {images}\n"
            f"Videos: {videos}\n"
            f"Text: {text}\n---"
        )

    return "\n".join(formatted_context)


def build_prompt(preprocessed_context, question):
    prompt = f"""You are a helpful assistant for answering questions about
      Amazon product reviews. Use the following retrieved context 
      to answer the question. If the context does not contain relevant information, 
      say you don't know.
      Instructions:
       - Use the retrieved context to answer the question.
       - If the context does not contain relevant information, say you don't know.

    Context:
      {preprocessed_context}
      Question: {question}
      Answer:"""
    return prompt


def gen_answer(prompt):
    # Call may return different shapes depending on client used (OpenAI SDK or a helper
    # that returns a Pydantic model). Handle both cases and normalize to a dict.
    response, raw_response = client.chat.completions.create_with_completion(
        model="gpt-4.1-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        response_model=RagGenerationResponse
    )

    return response, raw_response
   


def rag_pipeline(question, qdrant_client, top_k=5):
    retrieve_context = retrieve_data(question, qdrant_client=qdrant_client, k=top_k)
    preprocessed_context = process_context(retrieve_context)
    prompt = build_prompt(preprocessed_context, question)
    gen, raw_gen = gen_answer(prompt)

    # Normalize final answer whether gen is dict-like or an object
    if isinstance(gen, dict):
        answer_text = gen.get("text") or gen.get("answer") or str(gen)
        rag_generation_response = gen
    else:
        answer_text = getattr(gen, "answer", getattr(gen, "text", str(gen)))
        # try to convert to dict if pydantic object provides model_dump
        try:
            rag_generation_response = gen.model_dump() if hasattr(gen, "model_dump") else gen.dict()
        except Exception:
            rag_generation_response = str(gen)

    final_result = {
        "question": question,
        "answer": answer_text.answer if isinstance(answer_text, RagGenerationResponse) else answer_text,
        "retrieved_context": retrieve_context,
        "rag_generation_response": rag_generation_response,
    }

    return final_result


In [29]:
output = rag_pipeline("What are the some best laptops you would recommend?", qdrant_client=qdrant_client, top_k=5)

results.points:  [ScoredPoint(id=173, version=18, score=0.4465452, payload={'product_id': '8010052f-0ed9-427c-a98a-0467d5b8b1f1', 'text': 'CUK HP Pavilion 17 Touch Gaming Notebook (i7-7700HQ, 16GB DDR4 RAM, 1TB SSHD, NVIDIA GTX 1050 4GB, 17.3" Full HD Touchscreen, Windows 10) Best Laptop for Media Students Photographers Photoshop', 'description': [], 'images': [{'thumb': 'https://m.media-amazon.com/images/I/51bbXcvjqlL._AC_US40_.jpg', 'large': 'https://m.media-amazon.com/images/I/51bbXcvjqlL._AC_.jpg', 'variant': 'MAIN', 'hi_res': 'https://m.media-amazon.com/images/I/61TonVMnqKL._AC_SL1200_.jpg'}, {'thumb': 'https://m.media-amazon.com/images/I/41huf2znUWL._AC_US40_.jpg', 'large': 'https://m.media-amazon.com/images/I/41huf2znUWL._AC_.jpg', 'variant': 'PT01', 'hi_res': 'https://m.media-amazon.com/images/I/61kKkMY9JRL._AC_SL1200_.jpg'}, {'thumb': 'https://m.media-amazon.com/images/I/41V0IaCTEpL._AC_US40_.jpg', 'large': 'https://m.media-amazon.com/images/I/41V0IaCTEpL._AC_.jpg', 'variant':

In [30]:
output

{'question': 'What are the some best laptops you would recommend?',
 'answer': 'Based on the provided context, some of the best laptops recommended are: 1. CUK HP Pavilion 17 Touch Gaming Notebook - Features include i7-7700HQ processor, 16GB DDR4 RAM, 1TB SSHD, NVIDIA GTX 1050 4GB, 17.3" Full HD Touchscreen, Windows 10. It is suitable for media students, photographers, and Photoshop users. 2. HP 2023 Pavilion x360 14" FHD Touchscreen 2-in-1 Laptop - Features 12th Gen Intel Core i5-1235U (10 Cores), 8GB DDR4 RAM, 1TB PCIe SSD, WiFi 6, Backlit Keyboard, Fingerprint Reader, Windows 11. It has a 360° flip-and-fold convertible design. 3. ASUS N53SV-XE1 15.6-Inch Versatile Entertainment Laptop - Features Intel Core i7 2GHz, 4GB DDR3 RAM, 500GB 7200RPM Hard Drive, Nvidia GT540M graphics, Windows 7 Home Premium. It is designed for multimedia and entertainment. 4. HP 2018 Newest EliteBook Revolve 810 G3 Laptop - Features Intel Core i5-5200U, 8GB RAM, 128GB SSD, Windows 10 Pro, 11.6" HD LED-Back

In [31]:
print(output['answer'])

Based on the provided context, some of the best laptops recommended are: 1. CUK HP Pavilion 17 Touch Gaming Notebook - Features include i7-7700HQ processor, 16GB DDR4 RAM, 1TB SSHD, NVIDIA GTX 1050 4GB, 17.3" Full HD Touchscreen, Windows 10. It is suitable for media students, photographers, and Photoshop users. 2. HP 2023 Pavilion x360 14" FHD Touchscreen 2-in-1 Laptop - Features 12th Gen Intel Core i5-1235U (10 Cores), 8GB DDR4 RAM, 1TB PCIe SSD, WiFi 6, Backlit Keyboard, Fingerprint Reader, Windows 11. It has a 360° flip-and-fold convertible design. 3. ASUS N53SV-XE1 15.6-Inch Versatile Entertainment Laptop - Features Intel Core i7 2GHz, 4GB DDR3 RAM, 500GB 7200RPM Hard Drive, Nvidia GT540M graphics, Windows 7 Home Premium. It is designed for multimedia and entertainment. 4. HP 2018 Newest EliteBook Revolve 810 G3 Laptop - Features Intel Core i5-5200U, 8GB RAM, 128GB SSD, Windows 10 Pro, 11.6" HD LED-Backlit Touchscreen, Backlit keyboard, and spill-resistant. 5. HP 15.6" Micro-Edge H

Rag Pipeline with Grounding Context

In [32]:
class RAGUsedContext(BaseModel):
    id: str = Field(description="The ID of the retrieved review")
    review: str = Field(description="The text of the retrieved review")

class RagGenerationResponseReference(BaseModel):
    answer: str = Field(description="The answer to the question")
    reasoning: str = Field(description="The reasoning behind the answer")
    used_context: list[RAGUsedContext] = Field(description="The list of retrieved reviews used to generate the answer")
    references: list[RAGUsedContext] = Field(description="The list of references used to generate the answer")

In [36]:
def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )
    return response.data[0].embedding

def retrieve_data(query, qdrant_client=qdrant_client, k=5):
    query_embedding = get_embedding(query)
    results = qdrant_client.query_points(
        collection_name="Amazon_Electronics_Data_Collection",
        query=query_embedding,
        limit=k,
        with_payload=True,
    )

    print("results.points: ", results.points)

    retrieved_context_ids = []
    retrieve_context = []
    similarity_scores = []
    retrieved_context_rating_numbers = []
    retrieve_context_main_categories = []
    retrieve_context_prices = []
    retrieve_context_images = []
    retrieve_context_videos = []
    retrieve_context_stores = []
    retrieve_context_categories = []
    retrieve_context_details = []
    retrieve_context_descriptions = []
    retrieve_context_features = []

    for result in results.points:
        payload = result.payload or {}
        retrieved_context_ids.append(result.id)
        similarity_scores.append(result.score)
        retrieve_context.append(payload.get('text', ''))
        retrieved_context_rating_numbers.append(payload.get('rating_number', None))
        retrieve_context_main_categories.append(payload.get('main_category', None))
        retrieve_context_prices.append(payload.get('price', None))
        retrieve_context_images.append(payload.get('images', None))
        retrieve_context_videos.append(payload.get('videos', None))
        retrieve_context_stores.append(payload.get('store', None))
        retrieve_context_categories.append(payload.get('categories', None))
        retrieve_context_details.append(payload.get('details', None))
        retrieve_context_descriptions.append(payload.get('description', None))
        retrieve_context_features.append(payload.get('features', None))

    return {
        'retrieved_context_ids': retrieved_context_ids,
        'retrieve_context': retrieve_context,
        'similarity_scores': similarity_scores,
        'retrieved_context_rating_numbers': retrieved_context_rating_numbers,
        'retrieve_context_main_categories': retrieve_context_main_categories,
        'retrieve_context_prices': retrieve_context_prices,
        'retrieve_context_images': retrieve_context_images,
        'retrieve_context_videos': retrieve_context_videos,
        'retrieve_context_stores': retrieve_context_stores,
        'retrieve_context_categories': retrieve_context_categories,
        'retrieve_context_details': retrieve_context_details,
        'retrieve_context_descriptions': retrieve_context_descriptions,
        'retrieve_context_features': retrieve_context_features,
    }

def process_context(context):
    formatted_context = []
    count = len(context.get('retrieved_context_ids', []))

    for index in range(count):
        point_id = context['retrieved_context_ids'][index]
        text = context['retrieve_context'][index] if index < len(context.get('retrieve_context', [])) else ''
        score = context['similarity_scores'][index] if index < len(context.get('similarity_scores', [])) else None
        rating_number = context['retrieved_context_rating_numbers'][index] if index < len(context.get('retrieved_context_rating_numbers', [])) else None
        main_category = context['retrieve_context_main_categories'][index] if index < len(context.get('retrieve_context_main_categories', [])) else ''
        price = context['retrieve_context_prices'][index] if index < len(context.get('retrieve_context_prices', [])) else None
        images = context['retrieve_context_images'][index] if index < len(context.get('retrieve_context_images', [])) else []
        videos = context['retrieve_context_videos'][index] if index < len(context.get('retrieve_context_videos', [])) else []
        store = context['retrieve_context_stores'][index] if index < len(context.get('retrieve_context_stores', [])) else ''
        categories = context['retrieve_context_categories'][index] if index < len(context.get('retrieve_context_categories', [])) else []
        details = context['retrieve_context_details'][index] if index < len(context.get('retrieve_context_details', [])) else ''
        description = context['retrieve_context_descriptions'][index] if index < len(context.get('retrieve_context_descriptions', [])) else ''
        features = context['retrieve_context_features'][index] if index < len(context.get('retrieve_context_features', [])) else []

        formatted_context.append(
            f"ID: {point_id}\n"
            f"Score: {score}\n"
            f"Rating Number: {rating_number}\n"
            f"Main Category: {main_category}\n"
            f"Price: {price}\n"
            f"Store: {store}\n"
            f"Categories: {categories}\n"
            f"Details: {details}\n"
            f"Description: {description}\n"
            f"Features: {features}\n"
            f"Images: {images}\n"
            f"Videos: {videos}\n"
            f"Text: {text}\n---"
        )

    return "\n".join(formatted_context)

def build_prompt(preprocessed_context, question):
    prompt = f"""You are a helpful shopping assistant for answering questions about
      products in stock.
      You will be given a question and a lits of context

      Instructions:
      - You need to answer the question based on the provided context only
      - Never use word context and refer to it as the available products
      - As an output you need to provide:

      * The answer to the question based on the provided context
      * The list of the IDs of the chuns that were used to answer the question.
      only return the ones that are used in the answer.
      * Short description (1-2 sentences) of the item based on the description provided in the context

      - The short description should have the name of the item.
      - The answer to the question should contain detailed information about the product and returned with
      detailed specification in bullet points.

      Context:
        {preprocessed_context}
    Question: {question}
    """ 
    return prompt


def gen_answer(prompt):
    # Call may return different shapes depending on client used (OpenAI SDK or a helper
    # that returns a Pydantic model). Handle both cases and normalize to a dict.
    response, raw_response = client.chat.completions.create_with_completion(
        model="gpt-4.1-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        response_model=RagGenerationResponse,
    )

    return response, raw_response
   


def rag_pipeline(question, qdrant_client, top_k=5):
    retrieve_context = retrieve_data(question, qdrant_client=qdrant_client, k=top_k)
    preprocessed_context = process_context(retrieve_context)
    prompt = build_prompt(preprocessed_context, question)
    gen, raw_gen = gen_answer(prompt)

    # Normalize final answer whether gen is dict-like or an object
    if isinstance(gen, dict):
        answer_text = gen.get("text") or gen.get("answer") or str(gen)
        rag_generation_response = gen
    else:
        answer_text = getattr(gen, "answer", getattr(gen, "text", str(gen)))
        # try to convert to dict if pydantic object provides model_dump
        try:
            rag_generation_response = gen.model_dump() if hasattr(gen, "model_dump") else gen.dict()
        except Exception:
            rag_generation_response = str(gen)

    final_result = {
        "question": question,
        "original_output": answer_text,
        "raw_gen": raw_gen,
        "answer": answer_text.answer if isinstance(answer_text, RagGenerationResponse) else answer_text,
        "references": answer_text.references if isinstance(answer_text, RagGenerationResponseReference) else [],
        "retrieved_context_ids": retrieve_context['retrieved_context_ids'],
        "retrieved_context": retrieve_context,
        "similarity_scores": retrieve_context['similarity_scores'],
        "rag_generation_response": rag_generation_response,
    }

    return final_result


In [34]:
result = rag_pipeline("Can I get Some Good Yoga Mat?", qdrant_client=QdrantClient(url=qdrant_url, api_key=qdrant_api_key), top_k=10)

/var/folders/pw/cff5mdz55nb7ghs1f4rh8f9r0000gn/T/ipykernel_11338/3648419139.py:1: UserWarning: Api key is used with an insecure connection.
  result = rag_pipeline("Can I get Some Good Yoga Mat?", qdrant_client=QdrantClient(url=qdrant_url, api_key=qdrant_api_key), top_k=10)


results.points:  [ScoredPoint(id=1285, version=27, score=0.42909956, payload={'product_id': 'a1610625-949b-4d92-bc78-8d08ac33c2f6', 'text': 'Yoga Frog Sticker', 'description': ['Yoga Frog Sticker 4x4"'], 'images': [{'thumb': 'https://m.media-amazon.com/images/I/41twBrEokIL._AC_US40_.jpg', 'large': 'https://m.media-amazon.com/images/I/41twBrEokIL._AC_.jpg', 'variant': 'MAIN', 'hi_res': 'https://m.media-amazon.com/images/I/71gE5kQKJ6L._AC_SL1500_.jpg'}, {'thumb': 'https://m.media-amazon.com/images/I/31B94MWM6SL._AC_US40_.jpg', 'large': 'https://m.media-amazon.com/images/I/31B94MWM6SL._AC_.jpg', 'variant': 'PT01', 'hi_res': 'https://m.media-amazon.com/images/I/61hOWh6F2fL._AC_SL1260_.jpg'}], 'videos': [], 'price': 3.19, 'rating_number': 34, 'main_category': 'Computers', 'categories': ['Electronics', 'Computers & Accessories', 'Laptop Accessories', 'Skins & Decals'], 'store': 'Vinyl Junkie Graphics', 'details': {'Brand': 'Vinyl Junkie Graphics', 'Age Range (Description)': 'Teen', 'Reusabil

In [35]:
result

{'question': 'Can I get Some Good Yoga Mat?',
 'original_output': 'The available products do not include any yoga mats. However, there is a Yoga Frog Sticker (ID: 1285) which is a decorative item related to yoga, and a Yoga 720 15 Case (ID: 1213) which is a protective case for a Lenovo Yoga 720 laptop. If you are specifically looking for a yoga mat, it is not available among the listed products.',
 'raw_gen': ChatCompletion(id='chatcmpl-DhuXljJjEG3lTGxYf8bEzYyCHcujb', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_rXCoajPrclhhtTaORa40BMhA', function=Function(arguments='{"answer":"The available products do not include any yoga mats. However, there is a Yoga Frog Sticker (ID: 1285) which is a decorative item related to yoga, and a Yoga 720 15 Case (ID: 1213) which is a protective case for a Le